 「rank を変えると誤差と圧縮率がどう変わるか」を一覧にする

In [2]:
import torch
import torch.nn as nn
import pandas as pd

# ============================================================
# 目的: rank ごとに「誤差 vs 圧縮率」のトレードオフを表にする
# ============================================================
# 問い: rank を下げると誤差は増えるが、パラメータ数はどれだけ減るか？
#
# 元の 1 層:  Linear(784 → 512)          … 401,920 パラメータ
# 圧縮後 2 層: Linear(784 → r, bias=False)
#            + Linear(r → 512, bias=True) … 784*r + r*512 + 512
#
# compression_ratio = 元 / 圧縮後
#   > 1 → 圧縮できている（例: rank=64 なら約 4.8 倍）
#   < 1 → 逆にパラメータが増える（rank が大きすぎる場合）

torch.manual_seed(42)

# --- 準備（01 と同じ設定） ---
layer = nn.Linear(784, 512)
x = torch.randn(32, 784)
y_original = layer(x)

W = layer.weight.data   # (512, 784)
b = layer.bias.data     # (512,)
U, S, Vh = torch.linalg.svd(W, full_matrices=False)

# 試す rank のリスト（大きいほど精度↑、小さいほど圧縮↑）
ranks = [512, 256, 128, 64, 32, 16, 8]
results = []

original_params = W.numel() + b.numel()  # 512*784 + 512 = 401,920

for r in ranks:
    max_rank = min(W.shape)  # = 512。これ以上の rank は意味がない
    if r > max_rank:
        continue

    # --- rank r で SVD を打ち切り、近似重み W_r を作る ---
    U_r = U[:, :r]
    S_r = S[:r]
    Vh_r = Vh[:r, :]
    W_r = U_r @ torch.diag(S_r) @ Vh_r

    # --- 近似出力と誤差 ---
    y_approx = x @ W_r.T + b
    diff = y_original - y_approx

    mae = torch.mean(torch.abs(diff)).item()   # 平均絶対誤差
    mse = torch.mean(diff ** 2).item()         # 二乗平均誤差（01 と同じ指標）
    max_abs_error = torch.max(torch.abs(diff)).item()  # 最大ズレ

    # --- 2 層に置き換えたときのパラメータ数 ---
    # Linear(784, r, bias=False) → 784 * r
    # Linear(r, 512, bias=True)  → r * 512 + 512
    compressed_params = (W.shape[1] * r) + (r * W.shape[0]) + b.numel()
    compression_ratio = original_params / compressed_params

    results.append({
        "rank": r,
        "mae": mae,
        "mse": mse,
        "max_abs_error": max_abs_error,
        "original_params": original_params,
        "compressed_params": compressed_params,
        "compression_ratio": compression_ratio,
    })

df = pd.DataFrame(results)

# 列の読み方:
#   rank               … 使った特異値の個数
#   mse                … 小さいほど近似精度が高い
#   compressed_params  … 2 層化後のパラメータ数
#   compression_ratio  … 1 より大きい rank から圧縮効果が出る
df

,rank,mae,mse,max_abs_error,original_params,compressed_params,compression_ratio
0,512,7.507122e-07,8.869472e-13,0.000004,401920,664064,0.605243
1,256,1.900106e-01,5.696965e-02,0.987786,401920,332288,1.209553
2,128,3.118686e-01,1.526667e-01,1.703916,401920,166400,2.415385
3,64,3.818942e-01,2.295969e-01,1.961764,401920,83456,4.815951
4,32,4.186273e-01,2.746774e-01,2.344338,401920,41984,9.573171
5,16,4.401250e-01,3.032614e-01,2.335141,401920,21248,18.915663
6,8,4.506745e-01,3.194356e-01,2.421761,401920,10880,36.941176
